# Full baseline training: ABRobOcular Xception + ArcFace on the Kaggle VISOB copy

Trains the paper's recognition model (their `create_model`, their loader) on all 300 classes
(subject IDs 1141-3279 plus 7000) with the paper-style random 60/20/20 split.
Expected run time: about 3 to 3.5 hours for 10 epochs on one T4.

**Before "Save & Run All (Commit)":**
1. Accelerator = GPU (T4 x2 is fine), Internet = On.
2. Add-ons -> Secrets -> `github_token` is attached to this notebook.
3. Input panel -> the VISOB Augmented dataset (`mubashirulhassan00/visob-augmented-recent`) is attached.
4. The logging edit (`--verbose`, CSVLogger) is pushed to the branch set below.

Never import TensorFlow in a notebook cell: the Keras 2 switch only works in a fresh process, so all TF code runs via `!python`.

## 1. Setup: install tf-keras, clone the repo, record the commit

In [2]:
import os
BRANCH = "nandini"   # change to "main" once the training-script PR is merged

!pip install -q --no-deps tf-keras

from kaggle_secrets import UserSecretsClient
tok = UserSecretsClient().get_secret("github_token")
repo = "/tmp/ocular-adv-xai"
url = f"https://{tok}@github.com/nandininautiyal/ocular-adv-xai.git"
if not os.path.exists(repo):
    !git clone -q -b {BRANCH} {url} {repo}
    !git -C {repo} remote set-url origin https://github.com/nandininautiyal/ocular-adv-xai.git
else:
    !git -C {repo} pull {url} {BRANCH}
%cd {repo}
!git branch --show-current
!git log -1 --oneline | tee /kaggle/working/RUN_INFO.txt
assert "--verbose" in open("scripts/03_train_baseline.py").read(), \
    "Push the logging edit (CSVLogger + --verbose) to this branch first"

remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 468 bytes | 468.00 KiB/s, done.
From https://github.com/nandininautiyal/ocular-adv-xai
 * branch            nandini    -> FETCH_HEAD
Updating faf8531..7f19820
Fast-forward
 scripts/03_train_baseline.py | 7 +++++--
 1 file changed, 5 insertions(+), 2 deletions(-)
/tmp/ocular-adv-xai
nandini
7f19820 Add CSV logging and verbosity option to training script


## 2. Build the manifest and the 300-class 60/20/20 split (symlinks in /tmp, a minute or two)

In [3]:
!python scripts/01_build_manifest.py --out /kaggle/working/manifest_raw.csv
!python scripts/02_make_splits.py

Data root: /kaggle/input/datasets/mubashirulhassan00/visob-augmented-recent/VISOB_Augmented_Recent
Wrote 134258 rows to /kaggle/working/manifest_raw.csv
Skipped 0 files that did not match the filename pattern
classes: 300 | images: 69341

images per split:
 split
train         41607
test          13868
validation    13866

images per class per split (min / median / max):
split  test  train  validation
min     4.0   12.0         4.0
50%    48.0  144.0        48.0
max    75.0  226.0        75.0

fraction of test images whose (subject, condition, eye, phase) bucket also has training images: 98.2%
tree written to: /tmp/ocular_recognition
manifest written to: /kaggle/working/manifest_split.csv


## 3. Sanity checks before the long run

In [4]:
!nvidia-smi -L

import os
for s in ["train", "validation", "test"]:
    d = f"/tmp/ocular_recognition/{s}"
    n = sum(len(f) for _, _, f in os.walk(d))
    print(s, "| classes:", len(os.listdir(d)), "| images:", n)
# expected: 300 classes in each split; about 41607 / 13866 / 13868 images

GPU 0: Tesla T4 (UUID: GPU-ab9889a2-98b9-4d24-de08-3777ca0461e8)
GPU 1: Tesla T4 (UUID: GPU-e95f85c7-38d8-ef00-56f1-e5f55a083101)
train | classes: 300 | images: 41607
validation | classes: 300 | images: 13866
test | classes: 300 | images: 13868


## 4. Full training (about 3 to 3.5 hours for 10 epochs)

Settings follow the shipped reference config (10 epochs, batch 32, lr 1e-4).
The paper text says 100 epochs: that is a logged deviation, it would not fit one Kaggle session.
If this runs out of GPU memory, change `--batch-size 32` to 16 and log that as a second deviation.

In [5]:
!python -u scripts/03_train_baseline.py --split-dir /tmp/ocular_recognition --n-classes 300 --epochs 10 --tag full --verbose 2 2>&1 | tee /kaggle/working/train_full.log

TF 2.20.0 | tf_keras 2.20.1 | tf.keras comes from: tf_keras.src.engine.training | GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
Found 41607 images belonging to 300 classes.
Found 13866 images belonging to 300 classes.
Found 13868 images belonging to 300 classes.
classes found: 300 | train/val/test images: 41607/13866/13868
I0000 00:00:1791432304.539472     205 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1791432304.543476     205 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
83683744/83683744 [==============================] - 1s 0us/step
model parameters: 155,387,432
Epoch 1/10
I0000 00:00:1791432320.244667     

## 5. Summary of the run (reads the saved files, does not import TensorFlow)

In [6]:
import json, os
import pandas as pd

out = "/kaggle/working/trained_models"
p = f"{out}/results_full.json"
if os.path.exists(p):
    print(json.dumps(json.load(open(p)), indent=2))
else:
    print("results_full.json not found: the training cell did not finish, see train_full.log")

h = f"{out}/history_full.csv"
if os.path.exists(h):
    print(pd.read_csv(h).round(4).to_string())

!ls -lh /kaggle/working/trained_models

{
  "tag": "full",
  "n_classes": 300,
  "epochs_run": 10,
  "minutes": 173.0,
  "test_acc_margin_free": 0.979449091433516,
  "test_acc_with_labels": 0.9633689068358812,
  "checkpoint": "/kaggle/working/trained_models/Xception_ArcFace_full.keras"
}
      loss  accuracy  val_loss  val_accuracy      lr
0  31.1700    0.0013   18.0436        0.1866  0.0001
1  15.7623    0.1021    6.3142        0.7159  0.0001
2   6.5577    0.4836    3.4894        0.8863  0.0001
3   3.2882    0.7690    2.5215        0.9216  0.0001
4   2.1536    0.8650    1.9111        0.9449  0.0001
5   1.6975    0.8986    1.6994        0.9517  0.0001
6   1.3679    0.9185    1.9994        0.9485  0.0001
7   1.1321    0.9336    1.6212        0.9568  0.0001
8   0.9810    0.9433    1.2734        0.9665  0.0001
9   0.8891    0.9475    1.4785        0.9612  0.0001
total 1.7G
-rw-r--r-- 1 root root  851 Oct  8 06:58 history_full.csv
-rw-r--r-- 1 root root  888 Oct  8 06:58 log_full.csv
-rw-r--r-- 1 root root  248 Oct  8 07:04 resu